## Load latest headlines

In [1]:
# ========== Bootstrap: ensure required Python packages are present =========
import importlib, subprocess, sys
from typing import Optional

def _ensure(pkg_name: str, import_name: Optional[str] = None):
    try:
        importlib.import_module(import_name or pkg_name)
    except ModuleNotFoundError:
        subprocess.check_call([sys.executable, '-m', 'pip', 'install', pkg_name])
    finally:
        globals()[import_name or pkg_name] = importlib.import_module(import_name or pkg_name)

_ensure('pandas')
print('All dependencies ready.\n')


All dependencies ready.



In [2]:
from pathlib import Path
archive = Path('archive')
archive.mkdir(exist_ok=True)
import pandas as pd
latest = pd.read_csv('../headlines/latest.csv')
latest.head()

,pubdate,source,title,link
0,2026-11-25-06-49-00 +0000,wsj,Nordic Stocks Tipped to Open Slightly Higher,https://www.wsj.com/finance/stocks/nikkei-fall...
1,2026-11-25-06-43-00 +0000,nyt,"Abrego Garcia Voluntarily Leaves U.S., Ending ...",https://www.nytimes.com/2026/11/24/us/politics...
2,2026-11-25-06-41-18 +0000,bbc,Court to rule on South African president's bid...,https://www.bbc.co.uk/news/articles/cq4gn6gj3z...
3,2026-11-25-06-40-12 +0000,nypost,North Korea expels Swiss representative over s...,https://nypost.com/2026/11/25/world-news/north...
4,2026-11-25-06-36-00 +0000,bbc,"Kilmar Ábrego García leaves US for Spain, endi...",https://www.bbc.co.uk/news/articles/c6808pyy5d...


## Calculate word frequencies

In [3]:
import re
from collections import Counter
from datetime import datetime

with open('exclude.txt') as f:
    stop_words = set(w.strip() for w in f if w.strip())
words = re.findall(r'[A-Za-z]+', ' '.join(latest['title']).lower())
filtered = [w for w in words if w not in stop_words and len(w) > 1]
counts = Counter(filtered)
score_df = (
    pd.DataFrame(counts.items(), columns=['word','score'])
    .sort_values('score', ascending=False)
)
score_df[['score','word']].to_csv('scores.csv', index=False)
timestamp = datetime.utcnow().strftime('%Y-%m-%d-%H-00-00')
score_df[['score','word']].to_csv(archive / f'scores-{timestamp}.csv', index=False)
score_df.head()


/tmp/ipykernel_2611/2220774760.py:15: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  timestamp = datetime.utcnow().strftime('%Y-%m-%d-%H-00-00')


,word,score
184,nvidia,39
35,trump,37
403,huang,26
402,jensen,20
131,china,15


## Rank headlines by score

In [4]:
word_scores = dict(score_df[['word','score']].values)
latest['score'] = latest['title'].apply(
    lambda t: sum(
        word_scores.get(w.lower(), 0)
        for w in re.findall(r'[A-Za-z]+', t)
        if len(w) > 1
    )
)
ranked = latest.sort_values('score', ascending=False)
ranked[['score','pubdate','source','title','link']].to_csv('rank.csv', index=False)
ranked[['score','pubdate','source','title','link']].to_csv(archive / f'rank-{timestamp}.csv', index=False)
ranked.head()


,pubdate,source,title,link,score
94,2026-11-25-00-18-00 +0000,wsj,Jensen Huang will face a bipartisan maelstrom ...,https://www.wsj.com/politics/policy/jensen-hua...,157
250,2026-11-24-18-28-12 +0000,nypost,Trump says cheating Nvidia CEO Jensen Huang an...,https://nypost.com/2026/11/24/us-news/trump-sa...,152
382,2026-11-24-10-53-37 +0000,bbc,Trump faces backlash after saying he 'convince...,https://www.bbc.co.uk/news/articles/cq5y1k8ez4...,148
350,2026-11-24-12-00-00 +0000,wsj,Secretary of State Marco Rubio sounded alarms ...,https://www.wsj.com/politics/policy/rubio-soun...,147
412,2026-11-24-09-00-00 +0000,cbc,Why Trump's move to get Jensen Huang reinstate...,https://www.cbc.ca/news/world/trump-jensen-hua...,144


## Select top headlines

In [5]:
with open('exclude_sources.txt') as f:
    excluded_sources = {
        line.strip()
        for line in f
        if line.strip() and not line.strip().startswith('#')
    }

top_count = 10
top_rows = []
working = word_scores.copy()
remaining = latest.copy()

while len(top_rows) < top_count and not remaining.empty:
    scored_remaining = remaining.assign(
        score=remaining['title'].apply(
            lambda t: sum(
                working.get(w.lower(), 0)
                for w in re.findall(r'[A-Za-z]+', t)
                if len(w) > 1
            )
        )
    )
    ranked_loop = scored_remaining.sort_values('score', ascending=False)

    if ranked_loop.empty:
        break

    top_story = ranked_loop.iloc[0]
    remaining = remaining.drop(top_story.name)

    if top_story['source'] in excluded_sources:
        continue

    words = set(re.findall(r'[A-Za-z]+', top_story['title'].lower()))
    for word in words:
        working.pop(word, None)

    top_rows.append(top_story[['score', 'pubdate', 'source', 'title', 'link']])

top_df = pd.DataFrame(top_rows)
top_df.to_csv('top.csv', index=False)
top_df.to_csv(archive / f'top-{timestamp}.csv', index=False)
top_df


,score,pubdate,source,title,link
94,157,2026-11-25-00-18-00 +0000,wsj,Jensen Huang will face a bipartisan maelstrom ...,https://www.wsj.com/politics/policy/jensen-hua...
407,100,2026-11-24-09-45-00 +0000,wsj,Companies tied to an investment firm affiliate...,https://www.wsj.com/politics/policy/democrats-...
34,55,2026-11-25-03-00-00 +0000,wsj,A Democratic push to enact Medicare for All is...,https://www.wsj.com/politics/policy/democrats-...
101,51,2026-11-24-23-49-03 +0000,nypost,Hudson River Trading expanding at 3 World Tra...,https://nypost.com/2026/11/24/business/hudson-...
56,50,2026-11-25-02-00-00 +0000,wsj,What would happen to the U.S. economy if the A...,https://www.wsj.com/economy/worse-than-the-dot...
224,48,2026-11-24-20-06-02 +0000,latimes,A man's severed head was found in a rural Cali...,https://www.latimes.com/california/story/2026-...
55,44,2026-11-25-02-00-00 +0000,wsj,Pennsylvania Gov. Josh Shapiro is staking out ...,https://www.wsj.com/politics/policy/shapiro-ac...
27,43,2026-11-25-03-18-52 +0000,nypost,Elderly Alzheimer-stricken pianist crosses 2 s...,https://nypost.com/2026/11/24/us-news/elderly-...
95,41,2026-11-25-00-13-52 +0000,nypost,"Andrew Giuliani’s daughter, 4, steals spotligh...",https://nypost.com/2026/11/24/us-news/andrew-g...
360,37,2026-11-24-11-28-30 +0000,nypost,Two suspects arrested after break-in reported ...,https://nypost.com/2026/11/24/world-news/princ...


## Create JSON version of top headlines

In [6]:
import pandas as pd
pd.read_csv('top.csv').to_json('top.json', orient='records', indent=2)
